# Video AI Upscaler (Google Colab)
**Real-ESRGAN + NVIDIA GPU** | Upload, upscale to 4K/8K, preview in browser

### Setup
1. Go to **Runtime > Change runtime type > GPU (T4)**
2. Run all cells in order
3. Click the **ngrok** URL to open the web app

In [ ]:
# Cell 1: Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install opencv-python-headless numpy flask basicsr realesrgan pyngrok -q
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Patch basicsr compatibility (MUST run before importing basicsr)
import os, importlib
try:
    from torchvision.transforms.functional_tensor import rgb_to_grayscale
    print('No patch needed')
except ImportError:
    spec = importlib.util.find_spec('basicsr')
    if spec and spec.submodule_search_locations:
        deg_file = os.path.join(spec.submodule_search_locations[0], 'data', 'degradations.py')
        if os.path.exists(deg_file):
            with open(deg_file, 'r') as f:
                content = f.read()
            content = content.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            with open(deg_file, 'w') as f:
                f.write(content)
            print('Patched basicsr for torchvision compatibility')
    else:
        print('basicsr not found')

In [ ]:
# Cell 3: Verify patch works
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('All imports OK!')

In [ ]:
# Cell 4: Set your ngrok auth token
# Get a FREE token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # <-- Paste your token here

if not NGROK_AUTH_TOKEN:
    print("WARNING: Set your ngrok token above! Get one free at https://ngrok.com")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok token set!")

In [ ]:
# Cell 5: Create project directories
BASE_DIR = "/content/upscaler"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
MODEL_DIR = os.path.join(BASE_DIR, "models")
TEMPLATE_DIR = os.path.join(BASE_DIR, "templates")

for d in [INPUT_DIR, OUTPUT_DIR, MODEL_DIR, TEMPLATE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 6: (Optional) Mount Google Drive for large files
# Uncomment the lines below to mount your Drive

# from google.colab import drive
# drive.mount('/content/drive')
# # Then copy videos from Drive to input:
# # !cp "/content/drive/MyDrive/your_video.mp4" /content/upscaler/input/

In [ ]:
%%writefile /content/upscaler/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Video AI Upscaler</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: #0f0f1a; color: #e0e0e0; min-height: 100vh;
        }
        .container { max-width: 1100px; margin: 0 auto; padding: 30px 20px; }
        h1 {
            text-align: center; font-size: 2rem; margin-bottom: 8px;
            background: linear-gradient(135deg, #00d2ff, #7b2ff7);
            -webkit-background-clip: text; -webkit-text-fill-color: transparent;
        }
        .subtitle { text-align: center; color: #888; margin-bottom: 30px; font-size: 0.95rem; }
        .card {
            background: #1a1a2e; border-radius: 16px; padding: 30px;
            margin-bottom: 24px; border: 1px solid #2a2a4a;
        }
        .card-title { font-size: 1.1rem; font-weight: 600; margin-bottom: 20px; color: #fff; }
        .upload-area {
            border: 2px dashed #3a3a6a; border-radius: 12px; padding: 40px;
            text-align: center; cursor: pointer; transition: all 0.3s; position: relative;
        }
        .upload-area:hover, .upload-area.dragover { border-color: #7b2ff7; background: rgba(123,47,247,0.05); }
        .upload-area input[type=\"file\"] { position: absolute; inset: 0; opacity: 0; cursor: pointer; }
        .upload-icon { font-size: 3rem; margin-bottom: 12px; }
        .upload-text { font-size: 1rem; color: #aaa; }
        .upload-text strong { color: #7b2ff7; }
        .file-name { margin-top: 10px; font-size: 0.9rem; color: #00d2ff; word-break: break-all; }
        .options-row { display: flex; gap: 16px; margin-top: 20px; align-items: center; flex-wrap: wrap; }
        .scale-group { display: flex; gap: 8px; }
        .scale-btn {
            padding: 10px 24px; border: 2px solid #3a3a6a; border-radius: 10px;
            background: transparent; color: #ccc; font-size: 0.95rem; cursor: pointer; transition: all 0.2s;
        }
        .scale-btn:hover { border-color: #7b2ff7; color: #fff; }
        .scale-btn.active { border-color: #7b2ff7; background: rgba(123,47,247,0.2); color: #fff; font-weight: 600; }
        .btn {
            padding: 12px 32px; border: none; border-radius: 10px;
            font-size: 1rem; font-weight: 600; cursor: pointer; transition: all 0.2s;
        }
        .btn-primary { background: linear-gradient(135deg, #7b2ff7, #00d2ff); color: #fff; margin-left: auto; }
        .btn-primary:hover { opacity: 0.9; transform: translateY(-1px); }
        .btn-primary:disabled { opacity: 0.4; cursor: not-allowed; transform: none; }
        .btn-danger { background: #e74c3c; color: #fff; }
        .btn-danger:hover { background: #c0392b; }
        .progress-section { display: none; }
        .progress-section.visible { display: block; }
        .progress-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; }
        .progress-bar-wrapper { background: #2a2a4a; border-radius: 12px; height: 28px; overflow: hidden; position: relative; }
        .progress-bar { height: 100%; border-radius: 12px; background: linear-gradient(90deg, #7b2ff7, #00d2ff); transition: width 0.4s ease; }
        .progress-bar-text { position: absolute; inset: 0; display: flex; align-items: center; justify-content: center; font-size: 0.85rem; font-weight: 600; color: #fff; text-shadow: 0 1px 3px rgba(0,0,0,0.5); }
        .stats-row { display: flex; gap: 24px; margin-top: 16px; flex-wrap: wrap; }
        .stat { display: flex; flex-direction: column; gap: 2px; }
        .stat-label { font-size: 0.75rem; color: #888; text-transform: uppercase; letter-spacing: 0.5px; }
        .stat-value { font-size: 1.1rem; font-weight: 600; color: #fff; }
        .status-badge { display: inline-block; padding: 4px 14px; border-radius: 20px; font-size: 0.85rem; font-weight: 600; }
        .status-processing { background: rgba(123,47,247,0.2); color: #b388ff; }
        .status-done { background: rgba(0,210,100,0.2); color: #69f0ae; }
        .status-aborted { background: rgba(231,76,60,0.2); color: #ff8a80; }
        .status-error { background: rgba(231,76,60,0.2); color: #ff8a80; }
        .status-skipped { background: rgba(255,193,7,0.2); color: #ffd740; }
        .players-section { display: none; }
        .players-section.visible { display: block; }
        .players-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
        @media (max-width: 768px) { .players-grid { grid-template-columns: 1fr; } }
        .player-card { background: #12122a; border-radius: 12px; overflow: hidden; border: 1px solid #2a2a4a; }
        .player-label { padding: 12px 16px; font-weight: 600; font-size: 0.9rem; border-bottom: 1px solid #2a2a4a; }
        .player-label.input-label { color: #ffd740; }
        .player-label.output-label { color: #69f0ae; }
        .player-card video { width: 100%; display: block; background: #000; }
        .no-video { padding: 60px 20px; text-align: center; color: #555; font-size: 0.9rem; }
        .toast {
            position: fixed; bottom: 30px; right: 30px; padding: 14px 24px; border-radius: 10px;
            font-size: 0.95rem; font-weight: 500; z-index: 1000;
            transform: translateY(100px); opacity: 0; transition: all 0.3s;
        }
        .toast.show { transform: translateY(0); opacity: 1; }
        .toast-error { background: #e74c3c; color: #fff; }
        .toast-success { background: #27ae60; color: #fff; }
        .toast-info { background: #2980b9; color: #fff; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Video AI Upscaler</h1>
        <p class="subtitle">Real-ESRGAN + NVIDIA GPU (Colab) | Upload, upscale, and preview</p>
        <div class="card">
            <div class="card-title">Upload Video</div>
            <div class="upload-area" id="uploadArea">
                <input type="file" id="fileInput" accept="video/*">
                <div class="upload-icon">&#128253;</div>
                <div class="upload-text">Drag & drop a video or <strong>click to browse</strong></div>
                <div class="file-name" id="fileName"></div>
            </div>
            <div class="options-row">
                <div class="scale-group">
                    <button class="scale-btn active" data-scale="2" onclick="selectScale(2)">4K (2x)</button>
                    <button class="scale-btn" data-scale="4" onclick="selectScale(4)">8K (4x)</button>
                </div>
                <button class="btn btn-primary" id="uploadBtn" onclick="startUpload()" disabled>Start Upscaling</button>
            </div>
        </div>
        <div class="card progress-section" id="progressSection">
            <div class="progress-header">
                <div class="card-title" style="margin-bottom:0">Processing: <span id="procFilename"></span></div>
                <div style="display:flex;gap:12px;align-items:center">
                    <span class="status-badge status-processing" id="statusBadge">Processing</span>
                    <button class="btn btn-danger" id="abortBtn" onclick="abortJob()">Abort</button>
                </div>
            </div>
            <div class="progress-bar-wrapper">
                <div class="progress-bar" id="progressBar" style="width:0%"></div>
                <div class="progress-bar-text" id="progressText">0.0%</div>
            </div>
            <div class="stats-row">
                <div class="stat"><span class="stat-label">Frame</span><span class="stat-value" id="statFrame">0 / 0</span></div>
                <div class="stat"><span class="stat-label">Speed</span><span class="stat-value" id="statSpeed">-</span></div>
                <div class="stat"><span class="stat-label">ETA</span><span class="stat-value" id="statEta">--:--:--</span></div>
            </div>
        </div>
        <div class="card players-section" id="playersSection">
            <div class="card-title">Video Preview</div>
            <div class="players-grid">
                <div class="player-card">
                    <div class="player-label input-label">Original (Input)</div>
                    <div id="inputPlayerWrap"><div class="no-video">No video loaded</div></div>
                </div>
                <div class="player-card">
                    <div class="player-label output-label">Upscaled (Output)</div>
                    <div id="outputPlayerWrap"><div class="no-video">Not yet available</div></div>
                </div>
            </div>
        </div>
    </div>
    <div class="toast" id="toast"></div>
    <script>
        let selectedScale = 2, selectedFile = null, eventSource = null;
        let currentInputFile = '', currentOutputFile = '';
        function selectScale(s) {
            selectedScale = s;
            document.querySelectorAll('.scale-btn').forEach(b => b.classList.toggle('active', parseInt(b.dataset.scale) === s));
        }
        const fileInput = document.getElementById('fileInput');
        const uploadArea = document.getElementById('uploadArea');
        const fileNameEl = document.getElementById('fileName');
        const uploadBtn = document.getElementById('uploadBtn');
        fileInput.addEventListener('change', () => {
            if (fileInput.files.length > 0) {
                selectedFile = fileInput.files[0];
                fileNameEl.textContent = selectedFile.name + ' (' + (selectedFile.size/(1024*1024)).toFixed(1) + ' MB)';
                uploadBtn.disabled = false;
            }
        });
        uploadArea.addEventListener('dragover', e => { e.preventDefault(); uploadArea.classList.add('dragover'); });
        uploadArea.addEventListener('dragleave', () => uploadArea.classList.remove('dragover'));
        uploadArea.addEventListener('drop', e => {
            e.preventDefault(); uploadArea.classList.remove('dragover');
            if (e.dataTransfer.files.length > 0) {
                selectedFile = e.dataTransfer.files[0]; fileInput.files = e.dataTransfer.files;
                fileNameEl.textContent = selectedFile.name + ' (' + (selectedFile.size/(1024*1024)).toFixed(1) + ' MB)';
                uploadBtn.disabled = false;
            }
        });
        async function startUpload() {
            if (!selectedFile) return;
            uploadBtn.disabled = true;
            const form = new FormData(); form.append('video', selectedFile); form.append('scale', selectedScale);
            try {
                const res = await fetch('/upload', { method: 'POST', body: form });
                const data = await res.json();
                if (!res.ok) { showToast(data.error, 'error'); uploadBtn.disabled = false; return; }
                currentInputFile = data.input_file; currentOutputFile = data.output_file;
                if (data.status === 'skipped') {
                    showToast('Output already exists - skipped!', 'info');
                    showPlayers(currentInputFile, currentOutputFile); showProgressSkipped(currentInputFile);
                    uploadBtn.disabled = false; return;
                }
                showToast('Processing started!', 'success');
                showProgressSection(currentInputFile); showInputPlayer(currentInputFile); connectSSE();
            } catch (e) { showToast('Upload failed: ' + e.message, 'error'); uploadBtn.disabled = false; }
        }
        function connectSSE() {
            if (eventSource) eventSource.close();
            eventSource = new EventSource('/progress');
            eventSource.onmessage = (e) => updateProgress(JSON.parse(e.data));
            eventSource.onerror = () => setTimeout(() => { if (eventSource) eventSource.close(); eventSource = null; }, 2000);
        }
        function updateProgress(d) {
            document.getElementById('progressBar').style.width = d.progress + '%';
            document.getElementById('progressText').textContent = d.progress + '%';
            document.getElementById('statFrame').textContent = d.frame + ' / ' + d.total_frames;
            document.getElementById('statSpeed').textContent = d.fps_speed > 0 ? d.fps_speed + ' s/frame' : '-';
            document.getElementById('statEta').textContent = d.eta || '--:--:--';
            const badge = document.getElementById('statusBadge');
            badge.className = 'status-badge status-' + d.status;
            badge.textContent = d.status.charAt(0).toUpperCase() + d.status.slice(1);
            if (d.status === 'done') {
                document.getElementById('abortBtn').style.display = 'none'; uploadBtn.disabled = false;
                if (eventSource) { eventSource.close(); eventSource = null; }
                showPlayers(currentInputFile, currentOutputFile); showToast('Upscaling complete!', 'success');
            } else if (d.status === 'aborted' || d.status === 'error') {
                document.getElementById('abortBtn').style.display = 'none'; uploadBtn.disabled = false;
                if (eventSource) { eventSource.close(); eventSource = null; }
                showToast(d.status === 'aborted' ? 'Aborted.' : 'Error: ' + d.eta, d.status === 'aborted' ? 'info' : 'error');
            }
        }
        async function abortJob() {
            try { const r = await fetch('/abort', {method:'POST'}); const d = await r.json(); if(r.ok) showToast('Aborting...','info'); else showToast(d.error,'error'); }
            catch(e) { showToast('Abort failed','error'); }
        }
        function showProgressSection(fn) {
            document.getElementById('progressSection').classList.add('visible');
            document.getElementById('procFilename').textContent = fn;
            document.getElementById('abortBtn').style.display = 'inline-block';
            document.getElementById('progressBar').style.width = '0%';
            document.getElementById('progressText').textContent = '0.0%';
            document.getElementById('statFrame').textContent = '0 / 0';
            document.getElementById('statSpeed').textContent = '-';
            document.getElementById('statEta').textContent = '--:--:--';
            const b = document.getElementById('statusBadge'); b.className='status-badge status-processing'; b.textContent='Processing';
        }
        function showProgressSkipped(fn) {
            document.getElementById('progressSection').classList.add('visible');
            document.getElementById('procFilename').textContent = fn;
            document.getElementById('abortBtn').style.display = 'none';
            document.getElementById('progressBar').style.width = '100%';
            document.getElementById('progressText').textContent = '100%';
            const b = document.getElementById('statusBadge'); b.className='status-badge status-skipped'; b.textContent='Skipped';
        }
        function showInputPlayer(fn) {
            document.getElementById('playersSection').classList.add('visible');
            document.getElementById('inputPlayerWrap').innerHTML = '<video controls preload="metadata" src="/videos/input/' + encodeURIComponent(fn) + '"></video>';
            document.getElementById('outputPlayerWrap').innerHTML = '<div class="no-video">Processing... will appear when done</div>';
        }
        function showPlayers(inp, out) {
            document.getElementById('playersSection').classList.add('visible');
            document.getElementById('inputPlayerWrap').innerHTML = '<video controls preload="metadata" src="/videos/input/' + encodeURIComponent(inp) + '"></video>';
            document.getElementById('outputPlayerWrap').innerHTML = '<video controls preload="metadata" src="/videos/output/' + encodeURIComponent(out) + '"></video>';
        }
        function showToast(msg, type) {
            const t = document.getElementById('toast'); t.textContent = msg;
            t.className = 'toast toast-' + type + ' show'; setTimeout(() => t.classList.remove('show'), 4000);
        }
        async function restoreSession() {
            try {
                const res = await fetch('/status');
                const d = await res.json();
                if (d.status === 'idle') return;
                currentInputFile = d.filename;
                currentOutputFile = d.output_file;
                if (d.status === 'processing') {
                    showProgressSection(currentInputFile);
                    showInputPlayer(currentInputFile);
                    updateProgress(d);
                    connectSSE();
                    showToast('Reconnected to running job', 'info');
                } else if (d.status === 'done') {
                    showProgressSection(currentInputFile);
                    updateProgress(d);
                    document.getElementById('abortBtn').style.display = 'none';
                    showPlayers(currentInputFile, currentOutputFile);
                } else if (d.status === 'aborted' || d.status === 'error') {
                    showProgressSection(currentInputFile);
                    updateProgress(d);
                    document.getElementById('abortBtn').style.display = 'none';
                    if (currentInputFile) showInputPlayer(currentInputFile);
                }
            } catch (e) {}
        }
        restoreSession();
    </script>
</body>
</html>

In [ ]:
# Cell 8: Define and start Flask app IN-PROCESS (no subprocess)
import os, sys, json, time, threading, queue, cv2, torch
from flask import Flask, render_template, request, jsonify, Response, send_from_directory
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from pyngrok import ngrok

BASE_DIR = "/content/upscaler"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
MODEL_DIR = os.path.join(BASE_DIR, "models")
ALLOWED_EXT = {".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm"}

for d in [INPUT_DIR, OUTPUT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

app = Flask(__name__, template_folder=os.path.join(BASE_DIR, "templates"))
app.config['MAX_CONTENT_LENGTH'] = 4 * 1024 * 1024 * 1024  # 4GB max upload

current_job = {
    "running": False, "abort": False, "filename": "", "progress": 0.0,
    "frame": 0, "total_frames": 0, "fps_speed": 0.0, "eta": "",
    "status": "idle", "output_file": "",
}
progress_subscribers = []

def broadcast_progress():
    data = json.dumps(current_job)
    dead = []
    for q in progress_subscribers:
        try: q.put_nowait(data)
        except queue.Full: dead.append(q)
    for q in dead:
        if q in progress_subscribers: progress_subscribers.remove(q)

MODEL_URLS = {
    "RealESRGAN_x2plus": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "RealESRGAN_x4plus": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
}

def download_model(model_name):
    model_path = os.path.join(MODEL_DIR, f"{model_name}.pth")
    if os.path.exists(model_path): return model_path
    print(f"Downloading {model_name}...")
    import urllib.request
    urllib.request.urlretrieve(MODEL_URLS[model_name], model_path)
    print(f"Downloaded {model_name}")
    return model_path

def create_upscaler(scale):
    gpu_id = 0 if torch.cuda.is_available() else None
    if scale <= 2:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        model_path = download_model("RealESRGAN_x2plus"); netscale = 2
    else:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        model_path = download_model("RealESRGAN_x4plus"); netscale = 4
    return RealESRGANer(
        scale=netscale, model_path=model_path, model=model,
        tile=400, tile_pad=10, pre_pad=0,
        half=gpu_id is not None, gpu_id=gpu_id,
    )

def process_video(filename, scale):
    global current_job
    input_path = os.path.join(INPUT_DIR, filename)
    base, ext = os.path.splitext(filename)
    suffix = "4K" if scale == 2 else "8K"
    out_name = f"{base}_{suffix}{ext}"
    output_path = os.path.join(OUTPUT_DIR, out_name)
    current_job.update({"running": True, "abort": False, "filename": filename,
        "progress": 0.0, "frame": 0, "total_frames": 0, "fps_speed": 0.0,
        "eta": "", "status": "processing", "output_file": out_name})
    broadcast_progress()
    try:
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened(): raise RuntimeError(f"Cannot open {input_path}")
        w, h = int(cap.get(3)), int(cap.get(4))
        fps, total = cap.get(5), int(cap.get(7))
        out_w, out_h = w * scale, h * scale
        current_job["total_frames"] = total; broadcast_progress()
        print(f"Processing: {w}x{h} -> {out_w}x{out_h}, {total} frames")
        upscaler = create_upscaler(scale)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))
        if not writer.isOpened(): raise RuntimeError("Cannot create video writer")
        frame_times = []
        for i in range(total):
            if current_job["abort"]: current_job["status"] = "aborted"; break
            ret, frame = cap.read()
            if not ret: break
            t0 = time.time()
            try: output, _ = upscaler.enhance(frame, outscale=scale)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); upscaler.tile = max(128, upscaler.tile // 2)
                print(f"OOM - reduced tile to {upscaler.tile}")
                output, _ = upscaler.enhance(frame, outscale=scale)
            if output.shape[1] != out_w or output.shape[0] != out_h:
                output = cv2.resize(output, (out_w, out_h), interpolation=cv2.INTER_LANCZOS4)
            writer.write(output)
            ft = time.time() - t0; frame_times.append(ft)
            avg = sum(frame_times[-30:]) / len(frame_times[-30:])
            eta_s = avg * (total - i - 1)
            current_job.update({"frame": i+1, "progress": round((i+1)/total*100, 1),
                "fps_speed": round(ft, 2),
                "eta": f"{int(eta_s//3600):02d}:{int(eta_s%3600//60):02d}:{int(eta_s%60):02d}"})
            broadcast_progress()
        cap.release(); writer.release()
        if current_job["status"] == "aborted":
            if os.path.exists(output_path): os.remove(output_path)
            print("Aborted.")
        elif current_job["status"] == "processing":
            current_job["status"] = "done"; current_job["progress"] = 100.0
            print(f"Done! Output: {out_name}")
    except Exception as e:
        current_job["status"] = "error"; current_job["eta"] = str(e)
        print(f"Error: {e}")
    finally:
        current_job["running"] = False; broadcast_progress(); torch.cuda.empty_cache()

@app.route("/")
def index(): return render_template("index.html")

@app.route("/upload", methods=["POST"])
def upload():
    if current_job["running"]: return jsonify({"error": "A job is already running."}), 409
    file = request.files.get("video")
    if not file or file.filename == "": return jsonify({"error": "No file selected."}), 400
    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in ALLOWED_EXT: return jsonify({"error": f"Unsupported: {ext}"}), 400
    scale = int(request.form.get("scale", 2))
    file.save(os.path.join(INPUT_DIR, file.filename))
    base, fext = os.path.splitext(file.filename)
    suffix = "4K" if scale == 2 else "8K"
    out_name = f"{base}_{suffix}{fext}"
    if os.path.exists(os.path.join(OUTPUT_DIR, out_name)):
        return jsonify({"status": "skipped", "message": f"Already exists: {out_name}",
            "output_file": out_name, "input_file": file.filename})
    threading.Thread(target=process_video, args=(file.filename, scale), daemon=True).start()
    return jsonify({"status": "started", "message": f"Processing {file.filename}",
        "input_file": file.filename, "output_file": out_name})

@app.route("/abort", methods=["POST"])
def abort():
    if not current_job["running"]: return jsonify({"error": "No job running."}), 400
    current_job["abort"] = True
    return jsonify({"status": "aborting"})

@app.route("/status")
def status(): return jsonify(current_job)

@app.route("/progress")
def progress_stream():
    q = queue.Queue(maxsize=50); progress_subscribers.append(q)
    def stream():
        yield f"data: {json.dumps(current_job)}\n\n"
        try:
            while True:
                try: yield f"data: {q.get(timeout=30)}\n\n"
                except queue.Empty: yield ": keepalive\n\n"
        except GeneratorExit: pass
        finally:
            if q in progress_subscribers: progress_subscribers.remove(q)
    return Response(stream(), mimetype="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

@app.route("/videos/input/<path:filename>")
def serve_input(filename): return send_from_directory(INPUT_DIR, filename)

@app.route("/videos/output/<path:filename>")
def serve_output(filename): return send_from_directory(OUTPUT_DIR, filename)

# Kill any existing ngrok tunnels
ngrok.kill()

# Start Flask in a background thread (IN-PROCESS, not subprocess)
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, threaded=True, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(5)  # Wait for Flask to fully start

# Verify Flask is running
import urllib.request
try:
    r = urllib.request.urlopen("http://localhost:5000/status")
    print(f"Flask is running! Status: {r.read().decode()}")
except Exception as e:
    print(f"ERROR: Flask failed to start: {e}")
    raise

# Create ngrok tunnel
public_url = ngrok.connect(5000)
print("=" * 60)
print(f"  App is live! Open this URL in your browser:")
print(f"  {public_url}")
print("=" * 60)

In [ ]:
# Cell 9 (Optional): Copy output back to Google Drive
# Uncomment to copy all upscaled videos to your Drive

# !cp /content/upscaler/output/* "/content/drive/MyDrive/Upscaled Videos/"